In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import requests

In [2]:
equip = pd.read_csv("Data/vision_cleaned.csv")

In [3]:
sum(equip['Stock ID'].isna())       # na 131
sum(~equip['Stock ID'].isna())      # value 1833 

935

In [4]:
stock_ids_mask = ~equip['Stock ID'].isna()
stock_ids = equip['Stock ID'][stock_ids_mask]

In [5]:
stock_ids[:10]

81    BNNVRL0
82    B1YW440
83    2F5232S
84    B1Y9QS9
85    7108899
86    2002305
87    B92SR70
88    BD20RS5
89    B4BNMY3
90    B01FLQ6
Name: Stock ID, dtype: object

In [6]:
sym_to_indices = defaultdict(list)
for idx, sym in stock_ids.items():   # idx = original DataFrame index
    sym_to_indices[sym].append(idx)

unique_symbols = list(sym_to_indices.keys())
max_workers = 8

In [7]:
def parallel_run(arr_len, max_workers, unique_symbols, sym_to_indices):
    sector_arr = np.full(arr_len, None, dtype=object)
    industry_arr = np.full(arr_len, None, dtype=object)
    error_id_list = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch_sector_industry, sym): sym for sym in unique_symbols}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Fetching sectors/industries"):
            sym, sector, industry, err = fut.result()
            if (sector is None) or (industry is None):
                error_id_list.append((sym, err))
            # write the same result to all rows that have this symbol
            for idx in sym_to_indices[sym]:
                sector_arr[idx] = sector
                industry_arr[idx] = industry
                
    return sector_arr, industry_arr, error_id_list

# 4) worker that mimics original logic (uses .info)
def fetch_sector_industry(symbol):
    try:
        t = yf.Ticker(symbol)
        info = t.info
        # Explicitly raise if sector/industry missing
        sector = info.get('sector')
        industry = info.get('industry')
        if sector is None and industry is None:
            raise ValueError("No sector/industry returned")
        return symbol, sector, industry, None
    except Exception as e:
        return symbol, None, None, str(e)

In [8]:
# call yahoo finance api and retrieve sector and industry for each stock id
sector_arr, industry_arr, error_id_list = parallel_run(len(equip), max_workers, unique_symbols, sym_to_indices)


Fetching sectors/industries:   0%|          | 0/935 [00:00<?, ?it/s]HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
Fetching sectors/industries:   1%|          | 8/935 [00:02<03:20,  4.63it/s]HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
Fetching sectors/industries:   2%|▏         | 17/935 [00:05<03:27,  4.42it/s]HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
Fetching sectors/industries:   3%|▎         | 25/935 [00:07<02:58,  5.09it/s]HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
Fetching sectors/industries:   3%|▎         | 27/935 [00:08<03:58,  3.81it/s]HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
Fetching sectors/industries:   3%|▎         | 30/935 [00:09<04:11,  3.59it/s]HTTP

In [9]:
def yahoo_resolve_symbol(query, lang="en-US", region="AU"):
    """
    Resolve a query (SEDOL/ISIN/name/ticker) to a Yahoo symbol using Yahoo's search API.
    Returns the best match symbol (str) or None.
    """
    url = "https://query2.finance.yahoo.com/v1/finance/search"
    params = {"q": query, "lang": lang, "region": region}
    r = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    r.raise_for_status()
    data = r.json()
    quotes = data.get("quotes") or []
    if not quotes:
        return None

    # pick the first equity-like hit; adjust ranking logic as you like
    for q in quotes:
        sym = q.get("symbol")
        if sym:
            return sym
    return None

In [11]:
len(error_id_list)

935

In [10]:
# test yahoo resolve symbol with industry fetch func
for code, message in error_id_list[:10]:
    sym = yahoo_resolve_symbol(code)
    sym, sector, industry, err = fetch_sector_industry(sym)
    print(code, sym, sector, industry, err)
    

BNNVRL0 29M.AX Basic Materials Other Precious Metals & Mining None
B1YW440 III.L None None No sector/industry returned
B92SR70 ABBV Healthcare Drug Manufacturers - General None
2002305 ABT Healthcare Medical Devices None
7108899 ABBN.SW Industrials Electrical Equipment & Parts None
BD20RS5 AX1.AX Consumer Cyclical Apparel Retail None
B1Y9QS9 ABB.BO Industrials Specialty Industrial Machinery None
2F5232S None None None 'NoneType' object has no attribute 'upper'
B4BNMY3 ACN Technology Information Technology Services None
B1HHKD3 AER Industrials Rental & Leasing Services None


In [12]:
def resolve_fetch_combine(query):
    sym = yahoo_resolve_symbol(query)
    symbol, sector, industry, err = fetch_sector_industry(sym)
    return symbol, sector, industry, err 

def parallel_resolve_fetch(arr_len, max_workers, unique_symbols, sym_to_indices):
    sector_arr = np.full(arr_len, None, dtype=object)
    industry_arr = np.full(arr_len, None, dtype=object)
    error_id_list = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(resolve_fetch_combine, sym): sym for sym in unique_symbols}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Fetching sectors/industries"):
            sym, sector, industry, err = fut.result()
            if (sector is None) or (industry is None):
                error_id_list.append((sym, err))
            # write the same result to all rows that have this symbol
            for idx in sym_to_indices[sym]:
                sector_arr[idx] = sector
                industry_arr[idx] = industry
                
    return sector_arr, industry_arr, error_id_list

In [19]:
arr_len_1 = len(error_id_list[:10])
unique_symbols_1 = [sym for sym, err in error_id_list[:10]]
sym_to_indices_1 = {sym: sym_to_indices[sym] for sym in unique_symbols_1}


['BNNVRL0',
 'B1YW440',
 'B92SR70',
 '2002305',
 '7108899',
 'BD20RS5',
 'B1Y9QS9',
 '2F5232S',
 'B4BNMY3',
 'B1HHKD3']

In [30]:
sector_arr_1, industry_arr_1, error_id_list_1 = parallel_resolve_fetch(arr_len_1, max_workers, unique_symbols_1, sym_to_indices_1)

Fetching sectors/industries:   0%|          | 0/10 [00:00<?, ?it/s]


KeyError: None

In [23]:
# rerun parallel run but need to check sym to indices first

# Compose new sym_to_indices to map A to C
new_mapping = {b: sym_to_indices[a] for a, b in output.items() if a in sym_to_indices}

sector_arr, industry_arr, final_err_list  = parallel_run(len(error_id_list), max_workers, list(output.values()), new_mapping)

Fetching sectors/industries:   0%|          | 0/935 [00:00<?, ?it/s]

In [31]:
# insert new cols Sector & Industry right after Listed Country
cols = list(equip.columns)
insert_at = cols.index("Listed Country") + 1
equip.insert(insert_at, 'Sector', sector_arr)
equip.insert(insert_at + 1, 'Industry', industry_arr)

In [32]:
# check
equip[~equip['Stock ID'].isna()][:10]

,Effective Date,Fund Name,Option Name,Asset Class Name,Int/Ext,Name/Kind of Investment Item,Currency,Stock ID,Listed Country,Sector,Industry,Units Held,% Ownership,Address,Value (AUD),Weighting
57,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,3M Co,NaN,US88579Y1010,US,Industrials,Conglomerates,1999.97,NaN,NaN,416985.64,0.000153
58,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,3i Group PLC,NaN,GB00B1YW4409,GB,Financial Services,Asset Management,1675.37,NaN,NaN,121813.77,0.000045
59,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,A O Smith Corp,NaN,US8318652091,US,Industrials,Specialty Industrial Machinery,234.08,NaN,NaN,25787.63,0.000009
60,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,ABB Ltd,NaN,CH0012221716,CH,Industrials,Electrical Equipment & Parts,6460.99,NaN,NaN,565029.92,0.000207
61,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,ABN AMRO Bank NV,NaN,NL0011540547,NL,Financial Services,Banks - Diversified,834.25,NaN,NaN,20775.24,0.000008
62,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,ACS Actividades de Construccion y Servicio,NaN,ES0167050915,ES,Industrials,Engineering & Construction,1754.45,NaN,NaN,142134.78,0.000052
63,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,AECOM,NaN,US00766T1007,US,Industrials,Engineering & Construction,210.44,NaN,NaN,36305.99,0.000013
64,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,AES Corp/The,NaN,US00130H1059,US,Utilities,Utilities - Diversified,1030.33,NaN,NaN,21417.02,0.000008
65,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,AGC Inc,NaN,JP3112000009,JP,Industrials,Conglomerates,327.71,NaN,NaN,15894.44,0.000006
66,2024-12-31,Equip Super Fund,Balanced Growth,Listed Equity,1,AGL Energy Ltd,NaN,AU000000AGL7,AU,Utilities,Utilities - Independent Power Producers,230167.84,NaN,NaN,2598594.90,0.000953


In [25]:
#check
print(equip["Sector"].unique())
print(len(equip["Industry"].unique()))

array([None, 'Industrials', 'Financial Services', 'Utilities', 'Energy',
       'Consumer Cyclical', 'Technology', 'Communication Services',
       'Healthcare', 'Basic Materials', 'Consumer Defensive',
       'Real Estate', ''], dtype=object)

In [27]:
import requests
import yfinance as yf

def yahoo_resolve_symbol(query, lang="en-US", region="US"):
    """
    Resolve a query (SEDOL/ISIN/name/ticker) to a Yahoo symbol using Yahoo's search API.
    Returns the best match symbol (str) or None.
    """
    url = "https://query2.finance.yahoo.com/v1/finance/search"
    params = {"q": query, "lang": lang, "region": region}
    r = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    r.raise_for_status()
    data = r.json()
    quotes = data.get("quotes") or []
    if not quotes:
        return None

    # pick the first equity-like hit; adjust ranking logic as you like
    for q in quotes:
        sym = q.get("symbol")
        if sym:
            return sym
    return None


sedol = "B1YW440"  # your example

symbol = yahoo_resolve_symbol(sedol, region="AU")  # AU site uses same backend; region helps ranking
print("Resolved:", symbol)

if symbol:
    t = yf.Ticker(symbol)
    info = t.get_info()  # yfinance >= 0.2 prefers get_info over .info
    print(info.get("sector"), "-", info.get("industry"))
else:
    print("Could not resolve SEDOL to a Yahoo symbol.")

Resolved: III.L
Financial Services - Asset Management
